# EDA A1 – Jakość danych (wersja pandas)

Pytanie biznesowe: **czy możemy ufać tym danym?**

Pracujemy na tabelach `users`, `orders`, `order_items`, `products`, `inventory_items` i `events`
z `bigquery-public-data.thelook_ecommerce`.

* Wczytaj tylko potrzebne kolumny i zamień je na najmniejsze odpowiednie typy danych.
  Sprawdź zużycie pamięci.
* Czy w danych są braki? Które są oczekiwane (proces biznesowy), a które są problemem?
* Czy klucze główne są unikalne (duplikaty)?
* Czy znaczniki czasu zgadzają się ze statusami zamówień?
* Jak świeże są dane i czy są rekordy z przyszłymi datami?
* Czy wartości liczbowe (ceny, koszty) mają sens?

In [5]:
import pandas as pd
import numpy as np

from bq_utils import read_bq

## Wczytanie danych

* `read_bq()` działa jak `pd.read_csv(usecols=..., dtype=...)`, tylko czyta z BigQuery.
* Pomijamy kolumny z danymi osobowymi (imię, nazwisko, e-mail, adres) – nie są nam potrzebne,
  a dobry nawyk to nie pobierać danych osobowych "na zapas".
* `Int32` / `Int8` to typy całkowite, które dopuszczają braki (NULL); `category` – dla tekstu,
  który powtarza się wiele razy (status, kraj, kategoria).

In [6]:
# Users – specify columns to keep and datatypes to convert

users_cols = ["id", "age", "gender", "country", "city", "traffic_source", "created_at"]

users_dtypes = {"id": "Int32", "age": "Int8", "gender": "category",
                "country": "category", "city": "category", "traffic_source": "category"}

users = read_bq("users", usecols=users_cols, dtype=users_dtypes)

users.head()

Zapytanie przetworzy: 5.62 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,age,gender,country,city,traffic_source,created_at
0,81146,62,F,Brasil,null,Search,2025-05-05 15:58:00+00:00
1,56556,36,F,Brasil,null,Search,2026-01-30 03:19:00+00:00
2,2792,27,F,Brasil,null,Search,2026-10-27 06:00:53.805757+00:00
3,18073,17,M,Brasil,null,Search,2022-10-31 12:32:00+00:00
4,35509,41,F,Brasil,null,Search,2024-02-27 17:48:00+00:00


In [7]:
users.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   id              100000 non-null  Int32              
 1   age             100000 non-null  Int8               
 2   gender          100000 non-null  category           
 3   country         100000 non-null  category           
 4   city            100000 non-null  category           
 5   traffic_source  100000 non-null  category           
 6   created_at      100000 non-null  datetime64[us, UTC]
dtypes: Int32(1), Int8(1), category(4), datetime64[us, UTC](1)
memory usage: 2.0 MB


In [8]:
# Orders

orders_cols = ["order_id", "user_id", "status", "gender", "num_of_item",
               "created_at", "shipped_at", "delivered_at", "returned_at"]

orders_dtypes = {"order_id": "Int32", "user_id": "Int32", "status": "category",
                 "gender": "category", "num_of_item": "Int8"}

orders = read_bq("orders", usecols=orders_cols, dtype=orders_dtypes)

orders.info(memory_usage="deep")

Zapytanie przetworzy: 6.73 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<class 'pandas.DataFrame'>
RangeIndex: 124452 entries, 0 to 124451
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   order_id      124452 non-null  Int32              
 1   user_id       124452 non-null  Int32              
 2   status        124452 non-null  category           
 3   gender        124452 non-null  category           
 4   num_of_item   124452 non-null  Int8               
 5   created_at    124452 non-null  datetime64[us, UTC]
 6   shipped_at    81063 non-null   datetime64[us, UTC]
 7   delivered_at  43755 non-null   datetime64[us, UTC]
 8   returned_at   12396 non-null   datetime64[us, UTC]
dtypes: Int32(2), Int8(1), category(2), datetime64[us, UTC](4)
memory usage: 5.5 MB


In [9]:
# Order items

order_items_cols = ["id", "order_id", "user_id", "product_id", "inventory_item_id", "status",
                    "sale_price", "created_at", "shipped_at", "delivered_at", "returned_at"]

order_items_dtypes = {"id": "Int32", "order_id": "Int32", "user_id": "Int32",
                      "product_id": "Int32", "inventory_item_id": "Int32", "status": "category"}

order_items = read_bq("order_items", usecols=order_items_cols, dtype=order_items_dtypes)

order_items.info(memory_usage="deep")

Zapytanie przetworzy: 13.57 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<class 'pandas.DataFrame'>
RangeIndex: 180746 entries, 0 to 180745
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   id                 180746 non-null  Int32              
 1   order_id           180746 non-null  Int32              
 2   user_id            180746 non-null  Int32              
 3   product_id         180746 non-null  Int32              
 4   inventory_item_id  180746 non-null  Int32              
 5   status             180746 non-null  category           
 6   sale_price         180746 non-null  float64            
 7   created_at         180746 non-null  datetime64[us, UTC]
 8   shipped_at         117847 non-null  datetime64[us, UTC]
 9   delivered_at       63571 non-null   datetime64[us, UTC]
 10  returned_at        18039 non-null   datetime64[us, UTC]
dtypes: Int32(5), category(1), datetime64[us, UTC](4), float64(1)
memory usage: 11.4 MB


In [10]:
# Products – "name" zostawiamy jako tekst: prawie każda nazwa jest inna,
# więc typ category nic by nie oszczędził (category opłaca się przy wielu powtórzeniach)

products_cols = ["id", "name", "brand", "category", "department",
                 "cost", "retail_price", "distribution_center_id"]

products_dtypes = {"id": "Int32", "brand": "category", "category": "category",
                   "department": "category", "distribution_center_id": "Int8"}

products = read_bq("products", usecols=products_cols, dtype=products_dtypes)

products.info(memory_usage="deep")

Zapytanie przetworzy: 3.30 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<class 'pandas.DataFrame'>
RangeIndex: 29120 entries, 0 to 29119
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   id                      29120 non-null  Int32   
 1   name                    29118 non-null  str     
 2   brand                   29096 non-null  category
 3   category                29120 non-null  category
 4   department              29120 non-null  category
 5   cost                    29120 non-null  float64 
 6   retail_price            29120 non-null  float64 
 7   distribution_center_id  29120 non-null  Int8    
dtypes: Int32(1), Int8(1), category(3), float64(2), str(1)
memory usage: 2.4 MB


In [11]:
# Inventory items

inventory_cols = ["id", "product_id", "created_at", "sold_at", "cost",
                  "product_category", "product_department"]

inventory_dtypes = {"id": "Int32", "product_id": "Int32",
                    "product_category": "category", "product_department": "category"}

inventory = read_bq("inventory_items", usecols=inventory_cols, dtype=inventory_dtypes)

inventory.info(memory_usage="deep")

Zapytanie przetworzy: 26.03 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<class 'pandas.DataFrame'>
RangeIndex: 487914 entries, 0 to 487913
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   id                  487914 non-null  Int32              
 1   product_id          487914 non-null  Int32              
 2   created_at          487914 non-null  datetime64[us, UTC]
 3   sold_at             180746 non-null  datetime64[us, UTC]
 4   cost                487914 non-null  float64            
 5   product_category    487914 non-null  category           
 6   product_department  487914 non-null  category           
dtypes: Int32(2), category(2), datetime64[us, UTC](2), float64(1)
memory usage: 16.8 MB


In [12]:
# Events – 2,4 mln wierszy, więc wczytujemy 10% próbkę (TABLESAMPLE).
# Wnioski z tej tabeli to SZACUNKI – pełne liczby policzyliśmy wcześniej w SQL.

events_cols = ["id", "user_id", "session_id", "sequence_number", "created_at",
               "traffic_source", "event_type", "browser"]

events_dtypes = {"id": "Int32", "user_id": "Int32", "sequence_number": "Int16",
                 "traffic_source": "category", "event_type": "category", "browser": "category"}

events = read_bq("events", usecols=events_cols, dtype=events_dtypes, sample_pct=10)

events.info(memory_usage="deep")

Zapytanie przetworzy: 17.02 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


<class 'pandas.DataFrame'>
RangeIndex: 186345 entries, 0 to 186344
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   id               186345 non-null  Int32              
 1   user_id          99675 non-null   Int32              
 2   session_id       186345 non-null  str                
 3   sequence_number  186345 non-null  Int16              
 4   created_at       186345 non-null  datetime64[us, UTC]
 5   traffic_source   186345 non-null  category           
 6   event_type       186345 non-null  category           
 7   browser          186345 non-null  category           
dtypes: Int16(1), Int32(2), category(3), datetime64[us, UTC](1), str(1)
memory usage: 12.1 MB


In [13]:
# Store all tables in a dictionary – kolejne sprawdzenia robimy dla wszystkich tabel naraz

tables = {"users": users, "orders": orders, "order_items": order_items,
          "products": products, "inventory": inventory, "events (10%)": events}

## Braki danych

* Policz % braków w każdej kolumnie każdej tabeli.
* Pokaż tylko kolumny, w których braki występują, od największego udziału.

In [14]:
# Check missing value percentage
# pd.concat ze słownikiem tworzy jedną Series z dwupoziomowym indeksem (tabela, kolumna)

(pd.concat({name: df.isna().mean().mul(100) for name, df in tables.items()})
 .rename("null_pct")
 .to_frame()
 .query("null_pct > 0")
 .sort_values("null_pct", ascending=False)
 .round(2)
)

null_pct
orders       returned_at      90.04
order_items  returned_at      90.02
orders       delivered_at     64.84
order_items  delivered_at     64.83
inventory    sold_at          62.96
events (10%) user_id          46.51
orders       shipped_at       34.86
order_items  shipped_at       34.80
products     brand             0.08
             name              0.01

## Duplikaty klucza głównego

* Czy każdy klucz występuje dokładnie raz? Uwaga: w `orders` kluczem jest `order_id`, nie `id`.

In [15]:
# Compare row count with number of unique keys

keys = {"users": "id", "orders": "order_id", "order_items": "id",
        "products": "id", "inventory": "id", "events (10%)": "id"}

(pd.DataFrame({"rows": {name: len(df) for name, df in tables.items()},
               "unique_keys": {name: tables[name][key].nunique() for name, key in keys.items()}})
 .assign(duplicates=lambda x: x["rows"] - x["unique_keys"])
)

,rows,unique_keys,duplicates
users,100000,100000,0
orders,124452,124452,0
order_items,180746,180746,0
products,29120,29120,0
inventory,487914,487914,0
events (10%),186345,186345,0


## Spójność statusów i znaczników czasu

* Dla każdego statusu zamówienia policz, jaki odsetek zamówień ma `shipped_at`, `delivered_at`
  i `returned_at`. Spodziewamy się np., że "Processing" nie ma żadnej daty wysyłki,
  a "Returned" ma wszystkie trzy.

In [16]:
# Create boolean flags with assign, then average them by status (mean of True/False = % True)

(orders
 .assign(shipped=lambda x: x["shipped_at"].notna(),
         delivered=lambda x: x["delivered_at"].notna(),
         returned=lambda x: x["returned_at"].notna())
 .groupby("status", observed=True)
 .agg({"order_id": "count", "shipped": "mean", "delivered": "mean", "returned": "mean"})
 .rename(columns={"order_id": "orders"})
 .sort_values("orders", ascending=False)
 .round(2)
)

,orders,shipped,delivered,returned
status,,,,
Shipped,37308,1.0,0.0,0.0
Complete,31359,1.0,1.0,0.0
Processing,24755,0.0,0.0,0.0
Cancelled,18634,0.0,0.0,0.0
Returned,12396,1.0,1.0,1.0


## Świeżość danych i daty z przyszłości

* Dla każdej tabeli z `created_at` znajdź najstarszy i najnowszy rekord.
* Policz rekordy z datą późniejszą niż "teraz" i o ile dni wybiegają w przyszłość.

In [17]:
now = pd.Timestamp.now(tz="UTC")

dated = {name: df for name, df in tables.items() if "created_at" in df.columns}

created = pd.concat({name: df["created_at"] for name, df in dated.items()}, names=["table", None])

freshness = (created
             .groupby(level="table")
             .agg(oldest="min",
                  newest="max",
                  future_rows=lambda s: (s > now).sum())
             .assign(days_ahead=lambda x: (x["newest"] - now).dt.days)
            )

freshness

,oldest,newest,future_rows,days_ahead
table,,,,
events (10%),2019-01-02 01:10:00+00:00,2026-11-01 22:32:08.109317+00:00,121,3
inventory,2018-11-25 06:22:51+00:00,2026-11-01 10:14:02.557601+00:00,37,3
order_items,2019-01-14 05:00:37+00:00,2026-11-01 23:53:20+00:00,1607,3
orders,2019-01-14 06:24:26+00:00,2026-10-29 00:45:48.635019+00:00,0,-1
users,2019-01-02 02:36:00+00:00,2026-10-28 19:27:30.752168+00:00,0,-1


In [18]:
# Inspect a few future-dated order items – use @ to reference a Python variable inside query

order_items.query("created_at > @now").sort_values("created_at").head()

,id,order_id,user_id,product_id,inventory_item_id,status,sale_price,created_at,shipped_at,delivered_at,returned_at
43243,52551,36225,29110,18433,141690,Processing,23.99,2026-10-29 08:03:56.290316+00:00,NaT,NaT,NaT
17077,44514,30711,24619,25201,119941,Shipped,13.49,2026-10-29 08:09:54.906065+00:00,2026-10-26 10:12:09.906065+00:00,NaT,NaT
118414,42366,29225,23438,27435,114143,Shipped,55.00,2026-10-29 08:11:46+00:00,2026-10-27 21:05:25+00:00,NaT,NaT
2784,110100,75919,61031,15030,297280,Returned,7.46,2026-10-29 08:12:12.514305+00:00,2026-10-25 11:00:01.514305+00:00,2026-10-26 21:05:01.514305+00:00,2026-10-26 22:40:01.514305+00:00
11290,137148,94471,75829,28783,370226,Complete,11.50,2026-10-29 08:12:48+00:00,2026-10-28 10:34:21+00:00,2026-11-01 22:53:21+00:00,NaT


In [23]:
# Share of rows where shipped_at is earlier than created_at (orders vs order_items)
# Only rows with a shipping date – comparing with NaT always gives False

pd.DataFrame({
    name: {
        "shipped_rows": df["shipped_at"].notna().sum(),
        "shipped_before_created": (df["shipped_at"] < df["created_at"]).sum(),
    }
    for name, df in {"orders": orders, "order_items": order_items}.items()
}).T.assign(pct=lambda x: (x["shipped_before_created"] / x["shipped_rows"] * 100).round(2))

,shipped_rows,shipped_before_created,pct
orders,81063,0,0.00
order_items,117847,35863,30.43


## Wartości liczbowe

* Sprawdź rozkład cen i kosztów z `describe()`.
* Czy są produkty, których koszt jest wyższy niż cena katalogowa (sprzedaż ze stratą)?
* Czy są pozycje sprzedane za 0 lub mniej?

In [19]:
products[["cost", "retail_price"]].describe().round(2)

,cost,retail_price
count,29120.00,29120.00
mean,28.48,59.22
std,30.62,65.89
min,0.01,0.02
25%,11.28,24.00
50%,19.68,39.99
75%,34.44,69.95
max,557.15,999.00


In [20]:
order_items["sale_price"].describe().round(2)

count    180746.00
mean         59.58
std          66.44
min           0.02
25%          24.70
50%          39.99
75%          69.95
max         999.00
Name: sale_price, dtype: float64

In [21]:
# Products sold at a loss and items with non-positive price

products.query("cost > retail_price").shape[0], order_items.query("sale_price <= 0").shape[0]

(0, 0)

In [22]:
# Margin by department – first business-flavoured view of the data

(products
 .assign(margin_pct=lambda x: (x["retail_price"] - x["cost"]) / x["retail_price"] * 100)
 .groupby("department", observed=True)
 .agg({"id": "count", "retail_price": "mean", "margin_pct": "mean"})
 .rename(columns={"id": "products"})
 .round(1)
)

,products,retail_price,margin_pct
department,,,
Men,13131,63.2,50.7
Women,15989,55.9,51.4


## Wnioski (uzupełnij po uruchomieniu)

* Braki: które są oczekiwane (wynikają z procesu), a które są problemem jakości?
* Duplikaty: czy klucze są unikalne?
* Spójność: czy daty zgadzają się ze statusami?
* Świeżość: ile tabel ma rekordy z przyszłości i o ile dni?
* Wartości: czy ceny i marże mają sens?

Wnioski z wersji SQL są zapisane w CLAUDE.md (sekcja "Wnioski A1").

1. **Braki w datach są strukturalne, nie są błędami.** Puste `shipped_at` / `delivered_at` / `returned_at`
   odpowiadają dokładnie statusom zamówień (np. 90% braków `returned_at` = ~10% zwrotów).
   46,5% zdarzeń w `events` nie ma `user_id` (sesje niezalogowanych) → lejek liczymy po `session_id`.
2. **`order_items.created_at` jest niewiarygodne.** 30,4% wysłanych pozycji ma wysyłkę przed utworzeniem
   (w `orders`: 0%), a część rekordów ma daty do 3 dni w przyszłości.
   → Analizy w czasie opieramy na `orders.created_at`.
3. **Dane są kompletne i czyste, ale wyraźnie syntetyczne.** Brak duplikatów kluczy, brak cen ≤ 0
   i sprzedaży poniżej kosztu, statusy idealnie spójne z datami, marża ~51% w obu działach.
   Historia zmienia się z dnia na dzień (124 452 zamówienia dziś vs 125 470 w dniu 2026-10-21).